# Recreate the Hippo projection and reproduce Figures 3–4

**Run All on Sherlock in the original `sc-dl` environment on a high-memory compute node.**

This workflow starts from the surviving combined expression file:

```text
/oak/stanford/groups/engreitz/Users/tri/shared/20260325_Hippo_timecourse/Hippo_timecourse_combined.h5ad
```

The production notebook shows that the canonical object used versioned Ensembl IDs as `var_names` and stored gene symbols in `var["gene"]`. The surviving Oak copy currently lacks that column. This notebook therefore creates a **read-only-data-preserving metadata repair copy** on Oak: it copies the existing H5AD, restores only `var["gene"]` from an existing symbol column, symbol-like `var_names`, or GENCODE v43, and then validates the complete object. It never reads the deleted scratch sublibraries and never modifies the surviving source file.

After that repair, the notebook:

1. validates the combined query against the recovered production cohort;
2. recreates the final selected `top_specific_200` fixed-reference cNMF projection;
3. validates cell order, program order, CLR values, markers, conditions, and QC;
4. updates only the two active Hippo path overrides; and
5. executes Figures 3 and 4 in fresh kernels.

The newly generated query adapter, projection, receipts, executed notebooks, and figure copies are stored under the configured Oak rebuild directory.


In [ ]:
# Repository-path bootstrap for local support modules.
from pathlib import Path
import sys


def _find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "cc_perturbseq").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root containing src/cc_perturbseq. "
        "Run this notebook from within the cc-perturbseq-manuscript-analysis checkout."
    )


REPO_ROOT = _find_repo_root()
for _support_path in (REPO_ROOT, REPO_ROOT / "src"):
    _text = str(_support_path)
    if _text not in sys.path:
        sys.path.insert(0, _text)

print("Repository support paths added:", REPO_ROOT, "and", REPO_ROOT / "src")


## 1. Settings

The existing Oak H5AD is the immutable source. If it already contains a valid `var["gene"]`, it is used directly. Otherwise, a repaired copy is created at `REPAIRED_QUERY_H5AD`; all expression matrices, layers, observation metadata, guide matrices, and cell/feature order remain unchanged.

`STRICT_HISTORICAL = True` retains the recorded 212,891-cell cohort, 62,757 features, 54 sample counts, 17,480 reference-gene overlap, 7,268 selected projection genes, and 99,416 condition-specific QC-passing cells. A mismatch stops the workflow rather than silently changing the scientific input.


In [ ]:
from datetime import datetime, timezone
import gzip
import json
import os
import re
import shutil
import subprocess
import tempfile
import uuid

import numpy as np
import pandas as pd
from IPython.display import display

from src.cc_perturbseq.figure_notebook import get_path, load_paths
from src.cc_perturbseq.hippo_workflow import (
    check_runtime,
    execute_figure_notebooks,
    run_main_preflight,
    validate_hippo_panel_inputs,
    write_hippo_overrides,
)

config_file, paths = load_paths(REPO_ROOT)
REBUILD_ROOT = get_path(paths, "hippo_rebuild.output_root")
SPECTRA_TPM = get_path(paths, "hippo_rebuild.gene_spectra_tpm")
GENCODE_GTF = get_path(paths, "reference.gencode_v43_gtf")

SOURCE_QUERY_H5AD = Path(
    "/oak/stanford/groups/engreitz/Users/tri/shared/"
    "20260325_Hippo_timecourse/Hippo_timecourse_combined.h5ad"
)
REPAIRED_QUERY_H5AD = (
    REBUILD_ROOT / "Hippo_timecourse_combined.with_gene_symbols.h5ad"
)

OVERWRITE_METADATA_ADAPTER = False
OVERWRITE_PROJECTION = False
STRICT_HISTORICAL = True
COMPRESSION = None  # Projection output only; optionally "gzip" or "lzf".
RUN_FIGURE_NOTEBOOKS = True
FIGURE_NOTEBOOKS = ("Figure_03.ipynb", "Figure_04.ipynb")

PROJECTED_H5AD = (
    REBUILD_ROOT
    / "perturbseq_cNMF_projection"
    / "reprojection_gene_set_comparison"
    / "Hippo_timecourse_reprojected_cNMF_compare_gene_sets.cell_x_program.h5ad"
)
RUN_DIR = (
    REBUILD_ROOT
    / "runs"
    / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
)

display(
    {
        "configuration": str(config_file),
        "surviving_query_source": str(SOURCE_QUERY_H5AD),
        "metadata_repaired_query_if_needed": str(REPAIRED_QUERY_H5AD),
        "gencode_v43_gtf": str(GENCODE_GTF),
        "reference_spectra": str(SPECTRA_TPM),
        "projection_output": str(PROJECTED_H5AD),
        "run_record": str(RUN_DIR),
        "execute_figures": RUN_FIGURE_NOTEBOOKS,
    }
)


## 2. Check the environment and durable inputs

This stage checks the surviving H5AD, GENCODE v43 annotation, fixed cNMF gene-spectra TPM, runtime packages, and writable Oak destination before copying or projecting anything.


In [ ]:
versions = check_runtime(include_figures=RUN_FIGURE_NOTEBOOKS)

from src.cc_perturbseq.hippo_rebuild import validate_combined
from src.cc_perturbseq.hippo_projection import (
    project_top_specific_200,
    read_spectra,
    validate_projected,
)

for label, path in {
    "surviving combined query": SOURCE_QUERY_H5AD,
    "GENCODE v43 GTF": GENCODE_GTF,
    "fixed cNMF gene spectra": SPECTRA_TPM,
}.items():
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f"{label} is missing or empty: {path}")

reference = read_spectra(
    SPECTRA_TPM,
    strict_historical=STRICT_HISTORICAL,
)
REFERENCE_GENES = pd.Index(reference.columns.astype(str))
print("Reference shape:", reference.shape)
del reference

REBUILD_ROOT.mkdir(parents=True, exist_ok=True)
with tempfile.TemporaryFile(dir=REBUILD_ROOT) as probe:
    probe.write(b"Hippo output write check")

RUN_DIR.mkdir(parents=True, exist_ok=False)
display(versions)


## 3. Restore the production gene-symbol metadata when needed

The uploaded production notebook explicitly shows versioned Ensembl IDs as the feature index and symbols in `var["gene"]`. The Oak file is first inspected without loading `X`. Candidate symbol vectors are scored against the same fixed-reference gene list. The resolver uses, in order of evidence:

- `var["gene"]` or another explicit symbol/name column;
- symbol-like `var_names`; or
- GENCODE v43 mapping of versioned Ensembl IDs, with the stripped Ensembl ID retained only when no `gene_name` is available.

Under strict reproduction, the selected vector must recover exactly 17,480 genes shared with the fixed cNMF reference. If repair is required, the source H5AD is copied to Oak—using filesystem reflinks when supported—and only the copied `var` table is rewritten. The surviving source file is never modified.


In [ ]:
EXPECTED_REFERENCE_OVERLAP = 17_480
SYMBOL_COLUMNS = (
    "gene",
    "gene_symbol",
    "gene_name",
    "symbol",
    "feature_name",
)
ENSEMBL_COLUMNS = ("gene_id", "ensembl_id")
ENSEMBL_RE = re.compile(r"^ENSG\d+(?:\.\d+)?$")


def _file_record(path):
    path = Path(path).resolve()
    stat = path.stat()
    return {
        "path": str(path),
        "size_bytes": int(stat.st_size),
        "mtime_ns": int(stat.st_mtime_ns),
    }


def _anndata_io():
    try:
        from anndata.io import read_elem, write_elem
    except ImportError:
        from anndata._io.specs import read_elem, write_elem
    return read_elem, write_elem


def _read_var_only(path):
    import h5py

    read_elem, _ = _anndata_io()
    with h5py.File(path, "r") as handle:
        return read_elem(handle["var"])


def _clean_text(values):
    values = pd.Index(values).astype(str)
    return pd.Index(values.str.strip())


def _valid_symbols(values):
    values = _clean_text(values)
    return (
        len(values) > 0
        and not values.isna().any()
        and bool((values.str.len() > 0).all())
        and not bool(values.str.lower().isin({"nan", "none"}).any())
    )


def _ensembl_fraction(values):
    values = _clean_text(values)
    if len(values) == 0:
        return 0.0
    return float(
        np.mean([bool(ENSEMBL_RE.fullmatch(value)) for value in values])
    )


def _parse_gtf_gene_names(gtf_path):
    mapping = {}
    opener = gzip.open if str(gtf_path).endswith(".gz") else open
    pattern_id = re.compile(r'(?:^|;\s*)gene_id\s+"([^"]+)"')
    pattern_name = re.compile(r'(?:^|;\s*)gene_name\s+"([^"]+)"')

    with opener(gtf_path, "rt", encoding="utf-8") as handle:
        for line in handle:
            if not line or line.startswith("#"):
                continue
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 9 or fields[2] != "gene":
                continue
            match_id = pattern_id.search(fields[8])
            if match_id is None:
                continue
            gene_id = match_id.group(1).split(".", 1)[0]
            match_name = pattern_name.search(fields[8])
            gene_name = (
                match_name.group(1).strip()
                if match_name is not None
                else gene_id
            )
            mapping.setdefault(gene_id, gene_name or gene_id)

    if not mapping:
        raise ValueError(
            f"No gene_id/gene_name records were parsed from {gtf_path}"
        )
    return mapping


def _ensembl_id_vector(var):
    candidates = [("var_names", _clean_text(var.index))]
    for column in ENSEMBL_COLUMNS:
        if column in var.columns:
            candidates.append(
                (f"var[{column!r}]", _clean_text(var[column]))
            )
    return max(candidates, key=lambda item: _ensembl_fraction(item[1]))


def resolve_gene_symbols(var, reference_genes, gtf_path):
    """Resolve one symbol per feature and document why that vector was chosen."""
    reference_set = set(pd.Index(reference_genes).astype(str))
    candidates = []

    for column in SYMBOL_COLUMNS:
        if column not in var.columns:
            continue
        values = _clean_text(var[column])
        if _valid_symbols(values):
            candidates.append(
                {
                    "source": f"var[{column!r}]",
                    "symbols": values,
                    "reference_overlap": len(set(values) & reference_set),
                    "method": "existing feature metadata",
                }
            )

    index_values = _clean_text(var.index)
    if (
        _valid_symbols(index_values)
        and _ensembl_fraction(index_values) < 0.50
    ):
        candidates.append(
            {
                "source": "var_names",
                "symbols": index_values,
                "reference_overlap": len(set(index_values) & reference_set),
                "method": "symbol-like feature index",
            }
        )

    id_source, identifiers = _ensembl_id_vector(var)
    if _ensembl_fraction(identifiers) >= 0.50:
        mapping = _parse_gtf_gene_names(gtf_path)
        stripped = pd.Index(
            [value.split(".", 1)[0] for value in identifiers],
            dtype=object,
        )
        mapped = pd.Index(
            [mapping.get(identifier, identifier) for identifier in stripped],
            dtype=object,
        )
        if _valid_symbols(mapped):
            candidates.append(
                {
                    "source": f"GENCODE v43 mapped from {id_source}",
                    "symbols": mapped,
                    "reference_overlap": len(set(mapped) & reference_set),
                    "method": "GENCODE v43 gene_id to gene_name",
                }
            )

    if not candidates:
        raise ValueError(
            "Could not recover gene symbols from the H5AD feature metadata or "
            f"GENCODE annotation: {gtf_path}"
        )

    selected = max(
        candidates,
        key=lambda item: (
            item["reference_overlap"],
            item["source"] == "var['gene']",
        ),
    )
    selected["candidates"] = [
        {
            "source": item["source"],
            "method": item["method"],
            "reference_overlap": int(item["reference_overlap"]),
        }
        for item in candidates
    ]
    selected["reference_overlap"] = int(selected["reference_overlap"])
    selected["n_features"] = len(selected["symbols"])
    selected["n_unique_symbols"] = int(
        pd.Index(selected["symbols"]).nunique()
    )
    return selected


def _copy_file_reflink_first(source, destination):
    try:
        subprocess.run(
            [
                "cp",
                "--reflink=auto",
                "--sparse=always",
                str(source),
                str(destination),
            ],
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
        )
        return "cp --reflink=auto --sparse=always"
    except (FileNotFoundError, subprocess.CalledProcessError):
        shutil.copy2(source, destination)
        return "shutil.copy2 fallback"


def _rewrite_var_gene(path, symbols):
    import h5py

    read_elem, write_elem = _anndata_io()
    with h5py.File(path, "r+") as handle:
        var = read_elem(handle["var"])
        if len(var) != len(symbols):
            raise ValueError(
                f"Feature table has {len(var)} rows but symbol vector has {len(symbols)}"
            )
        var["gene"] = pd.Categorical(pd.Index(symbols).astype(str))
        del handle["var"]
        write_elem(handle, "var", var)
        handle.flush()


def _validate_gene_column(path, reference_genes):
    var = _read_var_only(path)
    if "gene" not in var.columns:
        raise ValueError(
            f"Metadata-adapted query still lacks var['gene']: {path}"
        )
    symbols = _clean_text(var["gene"])
    if not _valid_symbols(symbols):
        raise ValueError(
            f"Metadata-adapted query has invalid var['gene']: {path}"
        )
    overlap = len(
        set(symbols) & set(pd.Index(reference_genes).astype(str))
    )
    return {
        "n_features": len(symbols),
        "n_unique_symbols": int(symbols.nunique()),
        "reference_overlap": int(overlap),
    }


def ensure_query_gene_symbols(
    source_h5ad,
    repaired_h5ad,
    *,
    gtf_path,
    reference_genes,
    overwrite=False,
    strict_historical=True,
):
    source_h5ad = Path(source_h5ad).resolve()
    repaired_h5ad = Path(repaired_h5ad).resolve()
    gtf_path = Path(gtf_path).resolve()

    source_var = _read_var_only(source_h5ad)
    resolution = resolve_gene_symbols(
        source_var, reference_genes, gtf_path
    )
    expected_overlap = (
        EXPECTED_REFERENCE_OVERLAP if strict_historical else None
    )
    if (
        expected_overlap is not None
        and resolution["reference_overlap"] != expected_overlap
    ):
        raise ValueError(
            "Recovered gene-symbol vector does not match the production "
            "reference overlap: observed "
            f"{resolution['reference_overlap']}, expected {expected_overlap}. "
            f"Candidates: {resolution['candidates']}"
        )

    if (
        "gene" in source_var.columns
        and resolution["source"] == "var['gene']"
    ):
        return {
            "status": "source_has_valid_gene_symbols",
            "query_h5ad": str(source_h5ad),
            "source_h5ad": str(source_h5ad),
            "source_modified": False,
            "resolution": {
                key: value
                for key, value in resolution.items()
                if key != "symbols"
            },
        }

    receipt_path = repaired_h5ad.with_suffix(
        repaired_h5ad.suffix + ".metadata_adapter_receipt.json"
    )
    source_record = _file_record(source_h5ad)
    gtf_record = _file_record(gtf_path)

    if repaired_h5ad.exists() and not overwrite:
        if not receipt_path.is_file():
            raise FileExistsError(
                f"Existing repaired query lacks its receipt: {repaired_h5ad}. "
                "Set OVERWRITE_METADATA_ADAPTER=True only after inspecting it."
            )
        receipt = json.loads(receipt_path.read_text(encoding="utf-8"))
        if (
            receipt.get("source_file") != source_record
            or receipt.get("gtf_file") != gtf_record
        ):
            raise FileExistsError(
                "Existing repaired query was made from different source/GTF "
                "metadata. Set OVERWRITE_METADATA_ADAPTER=True to replace it."
            )
        validation = _validate_gene_column(
            repaired_h5ad, reference_genes
        )
        if (
            expected_overlap is not None
            and validation["reference_overlap"] != expected_overlap
        ):
            raise ValueError(
                f"Existing repaired query has reference overlap "
                f"{validation['reference_overlap']}; expected {expected_overlap}"
            )
        receipt["status"] = "reused_metadata_repaired_query"
        receipt["validation"] = validation
        return receipt

    repaired_h5ad.parent.mkdir(parents=True, exist_ok=True)
    temporary = repaired_h5ad.with_name(
        f".{repaired_h5ad.name}.{uuid.uuid4().hex}.tmp"
    )
    try:
        copy_method = _copy_file_reflink_first(
            source_h5ad, temporary
        )
        _rewrite_var_gene(temporary, resolution["symbols"])
        validation = _validate_gene_column(
            temporary, reference_genes
        )
        if (
            expected_overlap is not None
            and validation["reference_overlap"] != expected_overlap
        ):
            raise ValueError(
                f"Repaired query has reference overlap "
                f"{validation['reference_overlap']}; expected {expected_overlap}"
            )
        os.replace(temporary, repaired_h5ad)
    finally:
        temporary.unlink(missing_ok=True)

    receipt = {
        "status": "created_metadata_repaired_query",
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "query_h5ad": str(repaired_h5ad),
        "source_h5ad": str(source_h5ad),
        "source_modified": False,
        "source_file": source_record,
        "gtf_file": gtf_record,
        "copy_method": copy_method,
        "repair": "added only var['gene']; all other H5AD data copied unchanged",
        "resolution": {
            key: value
            for key, value in resolution.items()
            if key != "symbols"
        },
        "validation": validation,
    }
    receipt_path.write_text(
        json.dumps(receipt, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    return receipt


adapter_report = ensure_query_gene_symbols(
    SOURCE_QUERY_H5AD,
    REPAIRED_QUERY_H5AD,
    gtf_path=GENCODE_GTF,
    reference_genes=REFERENCE_GENES,
    overwrite=OVERWRITE_METADATA_ADAPTER,
    strict_historical=STRICT_HISTORICAL,
)
QUERY_H5AD = Path(adapter_report["query_h5ad"]).resolve()

(RUN_DIR / "query_metadata_adapter.json").write_text(
    json.dumps(adapter_report, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print("Effective query H5AD:", QUERY_H5AD)
display(adapter_report)


## 4. Validate the effective expression query

This runs the existing full structural and scientific validation after gene-symbol restoration. It checks the recorded shape and sample counts, raw-count totals, broad-QC constraints, layers, guide matrices, cell/feature identifiers, and the restored `var["gene"]`.


In [ ]:
query_validation = validate_combined(
    QUERY_H5AD,
    strict_historical=STRICT_HISTORICAL,
)
combined_report = {
    "status": "effective_combined_query_validated",
    "surviving_source_h5ad": str(SOURCE_QUERY_H5AD),
    "effective_query_h5ad": str(QUERY_H5AD),
    "metadata_adapter": adapter_report,
    "validation": query_validation,
}
(RUN_DIR / "combined_stage.json").write_text(
    json.dumps(combined_report, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
display(query_validation)


## 5. Recreate the selected fixed-reference projection

The projection retains the recovered comparison notebook’s exact `top_specific_200` ranking, duplicate-symbol summation, full-library CPM normalization, fixed-H multiplicative-update solve, usage normalization, and log2 CLR transform. It does not refit the reference basis.


In [ ]:
projection_report = project_top_specific_200(
    QUERY_H5AD,
    SPECTRA_TPM,
    PROJECTED_H5AD,
    overwrite=OVERWRITE_PROJECTION,
    strict_historical=STRICT_HISTORICAL,
    compression=COMPRESSION,
)
(RUN_DIR / "projection_stage.json").write_text(
    json.dumps(projection_report, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
display(projection_report["diagnostics"])
display(projection_report["validation"])


## 6. Validate the query/projection pair and reconnect Figures 3–4

The configuration is changed only after exact cell-order, program-order, CLR, marker, condition, and QC checks pass. The active machine-specific YAML is backed up and unrelated entries are preserved.


In [ ]:
pair_validation = validate_projected(
    QUERY_H5AD,
    PROJECTED_H5AD,
    strict_historical=STRICT_HISTORICAL,
)
panel_validation = validate_hippo_panel_inputs(QUERY_H5AD)
config_update = write_hippo_overrides(
    REPO_ROOT,
    QUERY_H5AD,
    PROJECTED_H5AD,
    backup_dir=RUN_DIR / "config_backups",
)

for name, payload in {
    "pair_validation.json": pair_validation,
    "panel_validation.json": panel_validation,
    "configuration_update.json": config_update,
}.items():
    (RUN_DIR / name).write_text(
        json.dumps(payload, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )

display(pair_validation)
display(panel_validation)
display(config_update)


## 7. Preflight and execute Figures 3 and 4

The existing main-figure preflight runs after the repaired query and projection are configured. Figures 3 and 4 then execute sequentially in fresh kernels using this exact Python environment. Executed notebooks, receipts, logs, and copies of successful figure outputs are retained on Oak.


In [ ]:
if RUN_FIGURE_NOTEBOOKS:
    preflight_report = run_main_preflight(
        REPO_ROOT,
        RUN_DIR / "main_figure_preflight.log",
    )
    figure_report = execute_figure_notebooks(
        REPO_ROOT,
        RUN_DIR,
        notebook_names=FIGURE_NOTEBOOKS,
    )
    display(figure_report)
else:
    print(
        "The validated query and projection are configured. "
        "Figure execution was disabled in settings."
    )

print("Surviving source query:", SOURCE_QUERY_H5AD)
print("Effective query used by projection and figures:", QUERY_H5AD)
print("Selected projection:", PROJECTED_H5AD)
print("Run records and completed figure copies:", RUN_DIR)
